# Rod-bundle PINN — step-by-step walkthrough

Trains the physics-informed neural network on your OpenFOAM data and checks it
against the 2023 DNS paper. Run the cells **top to bottom** (Shift+Enter).

It uses the same code as `src/train.py` and `src/evaluate.py`, so a result here is
identical to a terminal run. Kernel: pick **enygf**.

Before starting you need these files in `digitized_data/cfd_generated/`, all written by
`cfd/openfoam/extract_profiles.py`:
- `cfd_field.csv` — every CFD cell (the training data)
- `cfd_nusselt.csv` — the CFD's own Nusselt numbers
- `cfd_profiles.csv` — the sampled lines (for plots)

In [ ]:
import os, sys
from pathlib import Path

# Work from ml/ so the paths in configs/default.yaml resolve
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd() / "src"))

import yaml, torch, pandas as pd, matplotlib.pyplot as plt
import train as trainer
import evaluate as evaluator

cfg = yaml.safe_load(open("configs/default.yaml"))
print("working dir:", Path.cwd())
print("python:", sys.executable)
print("device:", "GPU (CUDA)" if torch.cuda.is_available() else "CPU")

## Step 1 — Look at the training data

The CFD profiles along the path the DNS paper plots on (`xi`): rod surface in the narrow
gap → gap centre → subchannel centre → back to the rod at 45°. Then the CFD's own Nusselt
numbers next to the DNS — this shows how far the RANS model itself is from the DNS,
before any machine learning.

In [ ]:
df = pd.read_csv(cfg["paths"]["cfd_profiles"])
field = pd.read_csv(cfg["paths"]["cfd_field"])
print(field.groupby(["kind", "quantity"]).size().rename("training points"))

path = df[df["line"].isin(["seg1", "seg2", "seg3"])]
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, q, label in zip(axes, ["w", "nut", "k"], ["axial velocity w [m/s]", "eddy viscosity nu_t [m2/s]", "TKE k [m2/s2]"]):
    g = path[path["quantity"] == q].sort_values("xi")
    ax.plot(g["xi"] / cfg["geometry"]["dh_dns"], g["value"], ".", ms=2)
    ax.set_xlabel("xi / Dh"); ax.set_title(label)
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, bc in zip(axes, ["isoT", "isoFlux"]):
    for case, g in path[(path["quantity"] == "T") & (path["bc"] == bc)].groupby("case_id"):
        g = g.sort_values("xi")
        ax.plot(g["xi"] / cfg["geometry"]["dh_dns"], g["value"], ".", ms=2, label=case)
    ax.set_xlabel("xi / Dh"); ax.set_title(f"temperature, {bc}"); ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

nu_cfd = pd.read_csv(cfg["paths"]["cfd_nusselt"])
dns = pd.read_csv(cfg["paths"]["dns_nusselt"]).melt(id_vars="Pr", value_vars=["Nu_isoT", "Nu_isoFlux"], var_name="bc", value_name="Nu_DNS")
dns["bc"] = dns["bc"].str.replace("Nu_", "")
t = nu_cfd.merge(dns, on=["Pr", "bc"], how="left")
t["CFD_vs_DNS"] = (t["Nu_CFD"] / t["Nu_DNS"] - 1).map(lambda v: f"{v:+.0%}" if v == v else "-")
t["Pr"] = t["Pr"].map("{:g}".format)
print("\nCFD Nusselt numbers vs. DNS (both with Dh = 0.0712 m):")
print(t[["Pr", "bc", "Nu_CFD", "Nu_DNS", "CFD_vs_DNS"]].to_string(index=False, float_format=lambda v: f"{v:.2f}"))

## Step 2 — Choose the run length

`QUICK = True` runs 100 epochs (~1 min) to check everything works.
`QUICK = False` runs the full schedule (6000 epochs, roughly 30–60 min on a laptop CPU).

In [ ]:
QUICK = False

if QUICK:
    cfg["training"].update(epochs_data=50, epochs_physics=50, ramp_epochs=20, print_every=10)
n = cfg["training"]["epochs_data"] + cfg["training"]["epochs_physics"]
print(f"{n} epochs")

## Step 3 — Train

- **Phase 1** (`epochs_data`): the network fits the CFD data — velocity, eddy viscosity, TKE,
  all 8 temperature fields, wall shear and wall heat flux — while keeping the bulk velocity
  at 1 m/s and zero gradients on the symmetry planes.
- **Phase 2** (`epochs_physics`): the governing equations (axial momentum, energy), the
  iso-flux wall condition and the overall force balance are switched on gradually, so the
  fields become physically consistent. Without this phase the pressure gradient and the
  iso-flux results come out wrong.

20% of the CFD cells are **held out**: the network never trains on them, and the error on
them at the end is the honest measure of accuracy.

In [ ]:
model = trainer.train(cfg)

## Step 4 — Loss curves

What to look for:
- **`data_*` curves fall** in phase 1 and stay low in phase 2.
- **Physics terms appear at the dotted line** (`pde_*`, `wall_heat_flux`, `wall_shear_balance`)
  and fall from there. They stay higher than the data terms; that is normal.
- **`dpdz`** (learned pressure gradient) jumps when physics switches on and should settle on
  the CFD value (black dashed line, computed from the CFD wall shear). The red dotted line is
  the DNS value — the gap between the two lines is a RANS-vs-DNS difference, not a PINN error.

In [ ]:
hist = pd.read_csv(cfg["paths"]["loss_history"])
terms = [c for c in hist.columns if c not in ("epoch", "phase", "total", "dpdz")]
switch = cfg["training"]["epochs_data"]

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
for t in terms:
    s = hist[["epoch", t]].dropna()
    axes[0].semilogy(s["epoch"], s[t].rolling(20, min_periods=1).mean(), label=t)
axes[0].axvline(switch, color="gray", ls=":")
axes[0].set_xlabel("epoch"); axes[0].set_title("loss terms (20-epoch moving average)")
axes[0].legend(fontsize=7, ncol=2)

# CFD pressure gradient from its wall shear: G * area = sum(tau_w) * rod arc length
import math
g = cfg["geometry"]
tau = field[field["quantity"] == "tau_w"]["value"].mean()
G_cfd = tau * (math.pi * g["rod_radius"] / 2) / (g["half_pitch"] ** 2 - math.pi * g["rod_radius"] ** 2 / 4)
axes[1].plot(hist["epoch"], hist["dpdz"], label="PINN")
axes[1].axhline(G_cfd, color="k", ls="--", label=f"OpenFOAM CFD ({G_cfd:.3f})")
axes[1].axhline(4 * 0.0637**2 / g["dh_cell"], color="r", ls=":", label="from DNS u_tau (0.207)")
axes[1].set_xlabel("epoch"); axes[1].set_title("learned pressure gradient dp/dz [m/s2]"); axes[1].legend()
plt.tight_layout(); plt.show()

## Step 5 — Evaluate

1. **Held-out CFD cells** — error on points the network never saw (`rmse_rel` = error / range
   of that field; below ~0.02 is a good fit).
2. **Along the sampled lines** — profile plots, PINN vs. CFD.
3. **Nusselt numbers** — PINN vs. the CFD it learned from vs. the DNS (Table 1 of the 2023 paper).
   `PINN_vs_CFD` measures the machine learning; `CFD_vs_DNS` measures the RANS model.
   The DNS was never used in training.
4. **Digitized DNS figures**, if you have filled in `digitized_data/dns_*.csv`.

In [ ]:
fit, nusselt, dns = evaluator.evaluate(cfg)

In [ ]:
from IPython.display import Image, display
for png in sorted(Path(cfg["paths"]["plots_dir"]).glob("*.png")):
    print(png.name)
    display(Image(filename=str(png)))

## Step 6 — Sparse-data experiment (the research question)

How much CFD data does the network need, and what do the physics terms add?
Each run trains on part of the data and is tested on the **same held-out cells**
and against the CFD Nusselt numbers (`docs/research_plan.md`).

- `PINN` = data + physics (the normal training). `plain NN` = the same network
  trained on data only (`use_physics: False`).
- `data_fraction` = share of the CFD cells used for training; `lines` = only the 4
  sampled lines (no wall data), the most sensor-like case.

Each run takes as long as Step 3 (PINN) or about a third of it (plain NN).
`RUN_ALL = False` runs the key pair (1% of the data); `True` runs all 8 rows
(several hours — leave it overnight). Finished runs are saved in
`outputs/experiments/` and skipped if you run the cell again, so an interrupted
batch just continues.

In [ ]:
import experiments as ex

RUN_ALL = False

runs = ex.RUNS if RUN_ALL else {
    "1% PINN": {"data_fraction": 0.01},
    "1% plain NN": {"data_fraction": 0.01, "use_physics": False},
}
table = ex.run_all(cfg, runs)

## If it doesn't converge well

Change one thing at a time in `configs/default.yaml` (or in `cfg` above) and rerun from Step 2:

- **Losses still falling at the end** → train longer: increase `epochs_physics`.
- **Curves noisy or jumping** → lower `lr` (e.g. `5.0e-4`).
- **Held-out error good but `PINN_vs_CFD` Nusselt off for iso-flux** → raise `wall_heat_flux`
  in `loss_weights`.
- **`dpdz` not settling on the CFD value** → raise `wall_shear_balance`.

Write down each change and its effect. That log is part of your methodology section.